# Vault RAG — Reranker (lekcja 17)

## Co się zmienia względem lekcji 14

Lekcja 14 (Hybrid) zwraca top-20 kandydatów z RRF i przekazuje je do recommend LLM.
Problem: BM25 wciąga do puli produkty nieistotne, co zalewa LLM szumem — Q01 traci produkt 12.

**Lekcja 17 dodaje reranker między safety filter a recommend LLM:**

```
Lekcja 14:  intent → hybrid Qdrant top-20 → safety filter → recommend LLM
Lekcja 17:  intent → hybrid Qdrant top-20 → safety filter → reranker top-5 → recommend LLM
```

### Jak działało wyszukiwanie w lekcji 14

Qdrant w lekcji 14 działał w dwóch krokach:

1. **Dense Prefetch**: query embed + product embed obliczane **niezależnie** → cosinus similarity → top-20
2. **Sparse Prefetch**: BM25 score (term overlap) → top-20
3. **RRF**: scala rankingi obu list → wynik końcowy `1/(60 + rank)`

Kluczowe ograniczenie: embed pytania jest obliczany raz, niezależnie od tego z jakim produktem
będzie porównywany. Model nie "widzi" konkretnego dokumentu podczas obliczania wektora pytania.

### Jak działa cross-encoder (reranker)

```
score = model(pytanie + dokument)   ← oba teksty razem w jednym forward pass
```

Model widzi jednocześnie i pytanie i dokument — może zauważyć że "codziennego mycia toalety"
w pytaniu odpowiada "do codziennego czyszczenia sanitariatów" w opisie produktu, podczas gdy
"Szkło Klar" który trafił do top-20 przez BM25 (słowo "kamień") nie ma nic wspólnego z toaletą.

Reranker jest wolniejszy (N wywołań dla N kandydatów, nie da się prekalkulować) ale precyzyjniejszy
dla małej puli. Stosujemy go TYLKO na ~15 kandydatach po safety filter, nie na 262 produktach.

### Model: BAAI/bge-reranker-v2-m3
Wielojęzyczny cross-encoder (obsługuje polski), działa lokalnie przez `sentence-transformers`.
Pobiera się raz (~1GB), nie wymaga API key.

In [1]:
%pip install qdrant-client openai instructor fastembed sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 8.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 46.5 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 58.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 69.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 57.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 59.8 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 49.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 56.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 68.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 28.9 MB/s  0:00:00
  Attempting uninstall: tokenizers━━╸━━━━━━━━━━━ 10/14 [scikit-learn]
    Found existing installation: tokenizers 0.23.1m━━━━━━━━━━━ 10/14 [scikit-learn]
    Uninstalling tokenizers-0.23.1:m╸━━━━━━━━━━━ 10/14 [s

In [2]:
import os
import re
import time
import json
from pathlib import Path
from datetime import datetime
from typing import List
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict

from openai import OpenAI
from pydantic import BaseModel, Field
import instructor
from fastembed import SparseTextEmbedding
from sentence_transformers import CrossEncoder
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, SparseVectorParams, SparseIndexParams,
    PointStruct, SparseVector,
    Filter, FieldCondition, MatchAny,
    Prefetch, Fusion, FusionQuery,
)

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "Brak klucza")

LLM_MODEL      = "google/gemini-3.1-flash-lite"
EMBED_MODEL    = "openai/text-embedding-3-small"
SPARSE_MODEL   = "Qdrant/bm25"
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"  # multilingual cross-encoder, works for Polish
RERANKER_TOP_K = 5                            # keep top-5 after reranking
COLLECTION     = "zurada_products_hybrid"

BASE_DIR       = Path(".")
VAULT_DIR      = BASE_DIR / "../11_vault_rag_improvements/zurada_vault"
QUESTIONS_FILE = BASE_DIR / "../5_navie_rag_extended_data/extended_golden_questions.json"
QDRANT_PATH    = BASE_DIR / "../14_qdrant_hybrid/qdrant_hybrid_storage"  # reuse existing index

_raw_client  = OpenAI(api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1")
llm          = instructor.from_openai(_raw_client, mode=instructor.Mode.JSON)
embed_client = _raw_client
bm25         = SparseTextEmbedding(model_name=SPARSE_MODEL)
qdrant       = QdrantClient(path=str(QDRANT_PATH))

# Load cross-encoder reranker — downloads ~1GB on first use
print(f"Loading reranker: {RERANKER_MODEL} ...")
reranker = CrossEncoder(RERANKER_MODEL, max_length=512)
print("Reranker loaded.")

with open(QUESTIONS_FILE, encoding="utf-8") as f:
    questions = json.load(f)

print(f"\nLLM:      {LLM_MODEL}")
print(f"Dense:    {EMBED_MODEL}")
print(f"Sparse:   {SPARSE_MODEL}")
print(f"Reranker: {RERANKER_MODEL}  top_k={RERANKER_TOP_K}")
print(f"Qdrant:   {QDRANT_PATH.resolve()}")
print(f"Questions: {len(questions)}")
print(f"API key:   {'OK' if OPENROUTER_API_KEY != 'Brak klucza' else 'BRAK'}")

Loading reranker: BAAI/bge-reranker-v2-m3 ...


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Reranker loaded.

LLM:      google/gemini-3.1-flash-lite
Dense:    openai/text-embedding-3-small
Sparse:   Qdrant/bm25
Reranker: BAAI/bge-reranker-v2-m3  top_k=5
Qdrant:   /Users/adamzurada/projects/ai-knowledge/rag-product-recommend/lessons/14_qdrant_hybrid/qdrant_hybrid_storage
Questions: 9
API key:   OK


---
## MAP_KNOWLEDGE + modele Pydantic

Identyczne jak w lekcji 11.

In [3]:
MAP_KNOWLEDGE = """
Vault Zurada to baza wiedzy produktów czyszczących, zorganizowana jako mapa połączonych stron.

Struktura vaultu:
- Powierzchnie/  — strony dla każdej powierzchni (np. toalety, podłogi, szkło)
- Kategorie/     — strony dla każdej kategorii (np. wc, zele-do-wc, odkamienianie)
- Metody/        — metody mycia (np. ręczne, myjnia, pianowanie)
- Produkty/      — pełne opisy produktów (frontmatter: product_id, ph, metody, kategorie)

Zasady odpowiedzi:
- Odpowiadaj WYŁĄCZNIE na podstawie dostarczonych stron produktów
- Nie wymyślaj cech produktów spoza dostarczonych danych
- Odróżniaj formy (żel vs piana), przeznaczenie (domowe vs przemysłowe), metodę (ręczne vs maszynowe)
"""


class ProductCandidate(BaseModel):
    product_id: int
    reason: str = Field(description="Jedno zdanie uzasadnienia")


class RecommendModel(BaseModel):
    chain_of_thought: List[str]         = Field(default_factory=list)
    product_ids: List[ProductCandidate] = Field(default_factory=list)
    excluded_product_ids: List[ProductCandidate] = Field(default_factory=list)



print("MAP_KNOWLEDGE and models defined")

MAP_KNOWLEDGE and models defined


---
## Krok 0 — Parsowanie metadanych produktów z vaultu

Czytamy strony produktów raz, wyciągamy:
- `kategorie` i `metody` z frontmattera
- `surfaces` (dozwolone) i `odradzane` z linków wiki w treści
- tekst opisu do osadzenia

Te metadane trafią jako **payload** do Qdrant.

In [4]:
def parse_frontmatter_list(md_text: str, field: str) -> list[str]:
    m = re.search(rf'^{field}:\s*\[(.*)\]', md_text, re.MULTILINE)
    if not m:
        return []
    return [v.strip().strip('"') for v in m.group(1).split(',') if v.strip()]


def parse_frontmatter_field(md_text: str, field: str) -> str | None:
    m = re.search(rf'^{field}:\s*(.+)$', md_text, re.MULTILINE)
    return m.group(1).strip().strip('"') if m else None


def parse_wiki_links(md_text: str, prefix: str) -> list[str]:
    """Extract link targets under a given Powierzchnie/ or Odradzane/ prefix."""
    pattern = rf'\[\[{re.escape(prefix)}/([^|\]]+)[|\]]'
    return re.findall(pattern, md_text)


def product_embed_text(md_text: str, name: str) -> str:
    """Build the text that will be embedded for this product."""
    title = parse_frontmatter_field(md_text, "title") or name
    parts = md_text.split("---", 2)
    body  = parts[2] if len(parts) >= 3 else md_text
    # Keep only ## Opis and ## Właściwości sections
    body  = re.sub(r'^\*\*[^*]+\*\*.*$', '', body, flags=re.MULTILINE)
    body  = re.sub(r'\[\[.*?\]\]', '', body)
    body  = re.sub(r'#+\s+', '', body)
    body  = re.sub(r'[⛔>]', '', body)
    body  = ' '.join(body.split())
    return f"{title}. {body}"


# Build product metadata dict
products: dict[str, dict] = {}

for f in sorted((VAULT_DIR / "Produkty").glob("*.md")):
    text = f.read_text(encoding="utf-8")
    pid  = parse_frontmatter_field(text, "product_id")
    if not pid:
        continue
    products[f.stem] = {
        "product_id": int(pid),
        "name":       f.stem,
        "title":      parse_frontmatter_field(text, "title") or f.stem,
        "kategorie":  parse_frontmatter_list(text, "kategorie"),
        "metody":     parse_frontmatter_list(text, "metody"),
        "surfaces":   parse_wiki_links(text, "Powierzchnie"),
        "odradzane":  parse_wiki_links(text, "Odradzane"),
        "embed_text": product_embed_text(text, f.stem),
    }

print(f"Parsed {len(products)} products")
# Quick sanity check
sample = list(products.values())[0]
print(f"Sample: {sample['name']}")
print(f"  kategorie: {sample['kategorie'][:3]}")
print(f"  surfaces:  {sample['surfaces'][:3]}")
print(f"  odradzane: {sample['odradzane'][:3]}")
print(f"  embed_text[:120]: {sample['embed_text'][:120]}")

Parsed 262 products
Sample: Zurada Agro Kompleks
  kategorie: ['chemia-do-maszyn-rolniczych', 'maszyny-rolnicze', 'mycie-maszyn']
  surfaces:  ['lakier samochodowy', 'metal', 'tworzywa sztuczne']
  odradzane: []
  embed_text[:120]: Zestaw do mycia maszyn. Zurada Agro Kompleks Zestaw do czyszczenia, odtłuszczania i zabezpieczania maszyn rolniczych. --


In [5]:
# Preview: what exactly goes into Qdrant for one product
import random

sample_name = "Zurada WC Żel Dom"  # change to any product name
p = products[sample_name]

print("=" * 70)
print(f"PRODUKT: {p['name']}")
print("=" * 70)

print("\n--- TEKST DO OSADZENIA (embed_text) ---")
print(p["embed_text"])

print("\n--- PAYLOAD (metadane w Qdrant) ---")
payload = {
    "name":      p["name"],
    "title":     p["title"],
    "kategorie": p["kategorie"],
    "surfaces":  p["surfaces"],
    "metody":    p["metody"],
    "odradzane": p["odradzane"],
}
for key, val in payload.items():
    print(f"  {key:12s}: {val}")

PRODUKT: Zurada WC Żel Dom

--- TEKST DO OSADZENIA (embed_text) ---
Żel do WC. Zurada WC Żel Dom Gęsty żel do czyszczenia toalet i ceramiki sanitarnej, usuwa kamień, rdzę oraz osady. --- - - - --- Opis Zurada Sanit Żel to skuteczny preparat do czyszczenia muszli WC oraz innych urządzeń sanitarnych. Gęsta formuła dobrze przylega do pionowych powierzchni, ułatwiając usuwanie kamienia, rdzy i osadów z twardej wody. Produkt pozostawia świeży, kwiatowy zapach i pomaga utrzymać higieniczną czystość w łazience. Przeznaczony jest do stosowania bezpośredniego, szczególnie w miejscach narażonych na osady mineralne. Właściwości Preparat działa na typowe zabrudzenia sanitarne, w tym kamień, rdzę i uporczywe naloty. Dzięki żelowej konsystencji dłużej utrzymuje się na czyszczonej powierzchni, zwiększając skuteczność działania.

--- PAYLOAD (metadane w Qdrant) ---
  name        : Zurada WC Żel Dom
  title       : Żel do WC
  kategorie   : ['zele-do-wc', 'wc', 'lazienka', 'czyszczenie-sanitarne', 'odk

---
## Krok 1 — Indeksowanie produktów w Qdrant (dense + sparse)

Kolekcja ma teraz **dwa wektory** per produkt:
- `dense` — embedding z OpenRouter (`text-embedding-3-small`, 1536 dims)
- `sparse` — BM25 z fastembed (`Qdrant/bm25`, liczba wymiarów zmienna, tylko niezerowe wartości)

BM25 działa lokalnie — żadne API key nie jest potrzebne.

In [6]:
def embed(texts: list[str]) -> list[list[float]]:
    resp = embed_client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [e.embedding for e in resp.data]


def embed_one(text: str) -> list[float]:
    return embed([text])[0]


def sparse_embed(texts: list[str]) -> list[dict]:
    """Return BM25 sparse vectors as {indices, values} dicts."""
    return [
        {"indices": e.indices.tolist(), "values": e.values.tolist()}
        for e in bm25.embed(texts)
    ]


def sparse_embed_one(text: str) -> dict:
    return sparse_embed([text])[0]


# Create hybrid collection if it doesn't exist yet
existing = [c.name for c in qdrant.get_collections().collections]

if COLLECTION not in existing:
    print(f"Creating hybrid collection '{COLLECTION}'...")
    qdrant.create_collection(
        collection_name=COLLECTION,
        vectors_config={
            "dense": VectorParams(size=1536, distance=Distance.COSINE),
        },
        sparse_vectors_config={
            "sparse": SparseVectorParams(index=SparseIndexParams()),
        },
    )

    product_list = list(products.values())
    BATCH = 50
    points = []

    for i in range(0, len(product_list), BATCH):
        batch   = product_list[i : i + BATCH]
        texts   = [p["embed_text"] for p in batch]
        dense_vecs  = embed(texts)
        sparse_vecs = sparse_embed(texts)

        for p, dvec, svec in zip(batch, dense_vecs, sparse_vecs):
            points.append(PointStruct(
                id     = p["product_id"],
                vector = {
                    "dense":  dvec,
                    "sparse": SparseVector(
                        indices=svec["indices"],
                        values=svec["values"],
                    ),
                },
                payload = {
                    "name":      p["name"],
                    "title":     p["title"],
                    "kategorie": p["kategorie"],
                    "surfaces":  p["surfaces"],
                    "metody":    p["metody"],
                    "odradzane": p["odradzane"],
                },
            ))
        print(f"  Indexed {min(i + BATCH, len(product_list))}/{len(product_list)}")

    qdrant.upsert(collection_name=COLLECTION, points=points)
    print(f"Done — {len(points)} products with dense+sparse vectors")
else:
    count = qdrant.count(collection_name=COLLECTION).count
    print(f"Collection '{COLLECTION}' already exists ({count} points) — skipping")

Collection 'zurada_products_hybrid' already exists (262 points) — skipping


---
## Krok 2 — Ekstrakcja intencji

Taki sam jak w lekcji 11 — LLM zwraca kategorie i powierzchnie jako closed-set enum.
Te wartości będą użyte jako filtr payload w Qdrant.

In [7]:
import enum as _enum


def _make_enum(name: str, keys: list[str]) -> type:
    return _enum.Enum(
        name,
        {k.upper().replace(" ", "_").replace("-", "_").replace("(", "").replace(")", ""): k
         for k in sorted(keys)},
        type=str,
    )


# Build enums from vault pages (needed to constrain intent extraction)
def _vault_page_names(subdir: str) -> list[str]:
    return [f.stem for f in (VAULT_DIR / subdir).glob("*.md")]


SurfaceEnum  = _make_enum("SurfaceEnum",  _vault_page_names("Powierzchnie"))
CategoryEnum = _make_enum("CategoryEnum", _vault_page_names("Kategorie"))
MethodEnum   = _make_enum("MethodEnum",   _vault_page_names("Metody"))


class IntentModel(BaseModel):
    surfaces: List[SurfaceEnum] = Field(
        default_factory=list,
        description="Powierzchnie fizyczne wymienione lub sugerowane w pytaniu. Domyślnie pusta lista.",
    )
    categories: List[CategoryEnum] = Field(
        default_factory=list,
        description="Kategorie produktów pasujące do pytania. Domyślnie pusta lista.",
    )
    methods: List[MethodEnum] = Field(
        default_factory=list,
        description="Metody aplikacji — tylko gdy pytanie jawnie wskazuje metodę. Domyślnie pusta lista.",
    )


INTENT_SYSTEM = """Jesteś analitykiem zapytań dla bazy wiedzy produktów czyszczących Zurada.

{MAP_KNOWLEDGE}

Na podstawie pytania klienta zidentyfikuj, które kategorie, powierzchnie i metody są potrzebne.
ZASADY:
1. UŻYWAJ WYŁĄCZNIE WARTOŚCI Z LISTY — nie wymyślaj własnych.
2. ZACHOWAJ ORYGINALNĄ PISOWNIĘ Z ENUMÓW.
3. 'surfaces' to fizyczne obiekty; 'categories' to grupy produktów.
Wybierz do 6 pozycji per pole."""


def extract_intent(question: str) -> tuple[IntentModel, object]:
    intent, completion = llm.chat.completions.create_with_completion(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": INTENT_SYSTEM.format(MAP_KNOWLEDGE=MAP_KNOWLEDGE)},
            {"role": "user",   "content": question},
        ],
        response_model=IntentModel,
        max_tokens=2000,
        temperature=0,
    )
    return intent, completion.usage


print(f"SurfaceEnum:  {len(list(SurfaceEnum))} values")
print(f"CategoryEnum: {len(list(CategoryEnum))} values")
print(f"MethodEnum:   {len(list(MethodEnum))} values")

# Demo
demo_q = questions[0]
intent_demo, _ = extract_intent(demo_q["question"])
print(f"\nDemo question: {demo_q['question']}")
print(f"Surfaces:   {[s.value for s in intent_demo.surfaces]}")
print(f"Categories: {[c.value for c in intent_demo.categories]}")

SurfaceEnum:  307 values
CategoryEnum: 285 values
MethodEnum:   22 values

Demo question: Potrzebuję zwykłego, gęstego żelu do codziennego mycia toalety w domu, żeby usunąć trochę kamienia i odświeżyć muszlę.
Surfaces:   ['toalety']
Categories: ['zele-do-wc', 'higiena-toalet', 'odkamienianie', 'dla-domu']


---
## Krok 3 — Hybrid retrieval (Prefetch + RRF)

### Co to jest Prefetch?

Prefetch to mechanizm Qdrant do opisu **wielu niezależnych sub-zapytań w jednym wywołaniu API**,
których wyniki są potem scalane przez zewnętrzny krok fuzji.

Bez Prefetch trzeba by zrobić dwa osobne requesty, odebrać dwie listy i scalić ręcznie w Pythonie.
Z Prefetch całość dzieje się wewnątrz Qdrant — jeden request, jeden response:

```
Jedno query_points() wywołanie:
│
├─ Prefetch #1: dense vector  → top-20  ← biegną wewnętrznie w Qdrant
├─ Prefetch #2: sparse vector → top-20  ← (potencjalnie równolegle)
│
└─ query=Fusion.RRF  →  scala oba top-20  →  zwraca jeden top-20
```

Każdy Prefetch ma własny `using` (który wektor), `limit` i opcjonalny `query_filter`.
Zewnętrzna fuzja widzi tylko połączone listy — nie wie jak działały sub-zapytania.

---

### Jak działa RRF i dlaczego K=60?

Wzór: `score(dokument) = Σ 1 / (K + rank)`

Stała `K=60` pochodzi z **oryginalnej pracy z 2009 roku** (Cormack, Clarke, Buettcher) i była
wyznaczona empirycznie na dziesiątkach różnych zbiorów danych TREC.

Co K kontroluje — intuicja przez liczby:

```
K=1  (agresywny):  rank 1 → 1/2  = 0.500,  rank 5 → 1/6  = 0.167,  rank 20 → 1/21  = 0.048
K=60 (łagodny):   rank 1 → 1/61 = 0.016,  rank 5 → 1/65 = 0.015,  rank 20 → 1/80  = 0.013
K=∞  (płaski):    każdy rank → taki sam score  (bez rankingu)
```

Przy `K=60` różnica między pozycją 1 a 20 wynosi tylko ~20% — RRF jest **demokratyczne**.
Dokument który jest na miejscu 3 w dense i miejscu 3 w sparse pokona dokument który jest na
miejscu 1 w dense ale nieobecny w sparse.

Przy małym K (np. K=1) produkt na pozycji 1 w dense zdominowałby wszystkich — wyniki byłyby
podobne do brania tylko najlepszego rankera.

**Dlaczego 60 a nie 50 czy 100?** Praktycznie każda wartość z zakresu 30–100 działa podobnie dobrze.
60 to historyczny default z papieru który „wszedł do standardu" — Qdrant, Elasticsearch i większość
frameworków używa go jako domyślnego. W praktyce nie warto go zmieniać bez konkretnego powodu i
eksperymentu.

📺 **Polecane wideo:** [Reciprocal Rank Fusion — wyjaśnienie wizualne](https://www.youtube.com/watch?v=6dDvfGrxFns)

In [8]:
def load_product_pages(names: list[str]) -> str:
    pages = []
    for name in names:
        path = VAULT_DIR / "Produkty" / f"{name}.md"
        if path.exists():
            pages.append(path.read_text(encoding="utf-8"))
    return "\n\n---\n\n".join(pages)


def qdrant_retrieve(
    question: str,
    intent: IntentModel,
    top_k: int = 20,
) -> tuple[str, list[str], list[float]]:
    """Hybrid search: dense + sparse via Prefetch + FusionQuery(RRF)."""
    q_dense  = embed_one(question)
    q_sparse = sparse_embed_one(question)

    # Payload filter — OR logic: kategorie OR surfaces from intent
    conditions = []
    if intent.categories:
        conditions.append(FieldCondition(
            key="kategorie",
            match=MatchAny(any=[c.value for c in intent.categories]),
        ))
    if intent.surfaces:
        conditions.append(FieldCondition(
            key="surfaces",
            match=MatchAny(any=[s.value for s in intent.surfaces]),
        ))
    qdrant_filter = Filter(should=conditions) if conditions else None

    def _search(filt):
        # Note: Prefetch uses `filter=`, top-level query_points uses `query_filter=`
        return qdrant.query_points(
            collection_name=COLLECTION,
            prefetch=[
                Prefetch(
                    query=q_dense,
                    using="dense",
                    filter=filt,
                    limit=top_k,
                ),
                Prefetch(
                    query=SparseVector(
                        indices=q_sparse["indices"],
                        values=q_sparse["values"],
                    ),
                    using="sparse",
                    filter=filt,
                    limit=top_k,
                ),
            ],
            query=FusionQuery(fusion=Fusion.RRF),  # wrap in FusionQuery per docs
            limit=top_k,
            with_payload=True,
        )

    response = _search(qdrant_filter)

    # Fallback: drop filter when too few results
    if len(response.points) < 3 and qdrant_filter is not None:
        response = _search(None)

    names  = [r.payload["name"] for r in response.points]
    scores = [round(r.score, 6) for r in response.points]
    return load_product_pages(names), names, scores


# Demo
context, names, scores = qdrant_retrieve(demo_q["question"], intent_demo)
print(f"Retrieved {len(names)} candidates (hybrid RRF):")
for name, score in zip(names, scores):
    pid = products.get(name, {}).get("product_id", "?")
    print(f"  [{pid:3}] {score:.6f}  {name}")

Retrieved 20 candidates (hybrid RRF):
  [139] 0.500000  Zurada WC Żel Dom
  [216] 0.500000  Zurada Powiew Radości
  [129] 0.342857  Zurada Szkło Klar
  [ 79] 0.333333  Zurada Biel Moc
  [190] 0.333333  Zurada Mydło Pomarańczowe
  [ 12] 0.302632  Zurada Sanit Żel
  [189] 0.250000  Zurada Kwiat Dłoni
  [ 46] 0.225490  Zurada Posadzka Lux
  [ 82] 0.200000  Zurada Toaleta Żel
  [229] 0.183333  Zurada Srebrny Blask
  [ 80] 0.166667  Zurada Kamień Gastro
  [211] 0.142857  Zurada Aloesowa Bryza
  [133] 0.125000  Zurada Kamień Łazienka
  [137] 0.125000  Zurada Pomarańczowe Dłonie
  [ 11] 0.122222  Zurada Czysta Łazienka
  [ 33] 0.112500  Zurada Kwas Pro
  [  2] 0.111111  Zurada Szklany Blask
  [188] 0.111111  Zurada Kwiatowa Pianka
  [ 29] 0.100000  Zurada Sanitar Błysk
  [ 68] 0.090909  Zurada Kwasowa Moc


---
## Krok 4 — Filtr bezpieczeństwa LLM

Identyczny jak w lekcji 11 (`SAFETY_SYSTEM_2`).

In [9]:
SAFETY_SYSTEM_2 = """Jesteś ekspertem chemicznym oceniającym bezpieczeństwo środków czyszczących.

Otrzymasz strony produktów z bazy Zurada oraz pytanie klienta.

Twoje jedyne zadanie: zidentyfikuj produkty NIEBEZPIECZNE dla opisanego kontekstu — nie filtruj
pod kątem trafności czy dopasowania (to robi osobny krok).

Wyklucz produkt (zwróć jego product_id) TYLKO jeśli zachodzi jeden z poniższych warunków:
1. Jego sekcja "Odradzane powierzchnie" zawiera powierzchnię WPROST wymienioną w pytaniu klienta.
2. Skład/opis wskazuje na KONKRETNĄ, BEZPOŚREDNIĄ niezgodność chemiczną z kontekstem
   (przykłady: silny kwas na marmur lub kamień naturalny, podchloryn na aluminium lub tkaniny).
3. Ostrzeżenia BHP tego produktu wprost wykluczają opisany sposób użycia.

WAŻNE — produkty higieny osobistej i mydła:
Mydła w płynie, żele do mycia rąk, szampony i środki do higieny osobistej (kategorie vault:
'mydla-w-plynie', 'higiena-rak', 'higiena-osobista', 'mycie-rak') są z definicji bezpieczne
dla skóry i typowych powierzchni łazienkowych. Wyklucz je wyłącznie gdy kryterium 1 jest
spełnione — tzn. gdy ich "Odradzane powierzchnie" zawiera konkretną powierzchnię z pytania.
Nie wykluczaj ich na podstawie kryterium 2 ani 3.

ZASADY OGÓLNE:
- Ten filtr ocenia wyłącznie BEZPIECZEŃSTWO, nie trafność produktu.
- Nie wykluczaj produktu tylko dlatego, że jest silny, profesjonalny lub nie najlepiej pasuje.
- Gdy wątpliwość — zachowaj produkt (nie wykluczaj)."""


class SafetyFilterModel(BaseModel):
    excluded_ids: List[int] = Field(default_factory=list)
    reasoning: List[str]   = Field(default_factory=list)


def filter_unsafe(
    question: str,
    product_context: str,
    product_names: list[str],
) -> tuple[str, list[str], SafetyFilterModel, object]:
    """Remove unsafe products; returns (filtered_context, filtered_names, result, usage)."""
    safety, completion = llm.chat.completions.create_with_completion(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": SAFETY_SYSTEM_2},
            {"role": "user",   "content": f"Strony produktów:\n\n{product_context}\n\n---\n\nPytanie klienta: {question}"},
        ],
        response_model=SafetyFilterModel,
        max_tokens=1000,
        temperature=0,
    )

    excluded_ids = set(safety.excluded_ids)
    id_map       = {p["product_id"]: p["name"] for p in products.values()}
    excluded_names = {id_map[pid] for pid in excluded_ids if pid in id_map}

    safe_names   = [n for n in product_names if n not in excluded_names]
    safe_context = load_product_pages(safe_names)
    return safe_context, safe_names, safety, completion.usage

---
## Krok 4.5 — Reranker

Cross-encoder ocenia każdą parę `(pytanie, embed_text produktu)` razem.
Widzi relacje między słowami w pytaniu i w opisie — nie tylko podobieństwo wektorów.

Wejście: ~15 kandydatów po filtrze bezpieczeństwa  
Wyjście: top-5 według score cross-encodera  

Przekazujemy `embed_text` (czysty opis) zamiast pełnego `.md` — mieści się w 512 tokenach.
Pełne strony `.md` wysyłamy do recommend LLM dopiero po selekcji rerankerem.

In [10]:
def rerank(
    question: str,
    candidate_names: list[str],
    top_k: int = RERANKER_TOP_K,
) -> tuple[list[str], list[float]]:
    """Score each (question, product_embed_text) pair with the cross-encoder.
    Returns (reranked_names, scores) sorted best-first, truncated to top_k.
    """
    if not candidate_names:
        return [], []

    # Build (query, document) pairs — use embed_text, not full .md
    pairs = [
        (question, products[name]["embed_text"])
        for name in candidate_names
        if name in products
    ]

    # Cross-encoder forward pass: one score per (question, product) pair
    scores = reranker.predict(pairs).tolist()

    # Sort by score descending, keep top_k
    ranked = sorted(zip(candidate_names, scores), key=lambda x: x[1], reverse=True)
    ranked = ranked[:top_k]

    names  = [n for n, _ in ranked]
    scores = [round(s, 4) for _, s in ranked]
    return names, scores


# Demo
demo_q = questions[0]
intent_demo, _ = extract_intent(demo_q["question"])
_, raw_names, _ = qdrant_retrieve(demo_q["question"], intent_demo)
_, safe_names, _, _ = filter_unsafe(demo_q["question"], load_product_pages(raw_names), raw_names)

reranked_names, reranked_scores = rerank(demo_q["question"], safe_names)

print(f"Pytanie: {demo_q['question']}")
print(f"After safety: {len(safe_names)} candidates")
print(f"After rerank: {len(reranked_names)} candidates (top-{RERANKER_TOP_K})")
print()
print(f"{'Rank':>4}  {'Score':>8}  {'ID':>4}  {'Nazwa'}")
print("─" * 60)
for rank, (name, score) in enumerate(zip(reranked_names, reranked_scores), 1):
    pid = products.get(name, {}).get("product_id", "?")
    tag = "  ← EXPECTED" if pid in demo_q["expectedProductIds"] else ""
    print(f"  {rank:2d}  {score:8.4f}  [{pid:3}]  {name}{tag}")

Pytanie: Potrzebuję zwykłego, gęstego żelu do codziennego mycia toalety w domu, żeby usunąć trochę kamienia i odświeżyć muszlę.
After safety: 17 candidates
After rerank: 5 candidates (top-5)

Rank     Score    ID  Nazwa
────────────────────────────────────────────────────────────
   1    0.9099  [139]  Zurada WC Żel Dom  ← EXPECTED
   2    0.6707  [ 12]  Zurada Sanit Żel  ← EXPECTED
   3    0.4773  [ 82]  Zurada Toaleta Żel  ← EXPECTED
   4    0.0189  [ 11]  Zurada Czysta Łazienka
   5    0.0045  [133]  Zurada Kamień Łazienka


---
## Krok 5 — Rekomendacja

Identyczna jak w lekcji 11 (`RECOMMEND_SYSTEM_3`).

In [11]:
RECOMMEND_SYSTEM_3 = """Jesteś ekspertem ds. środków czystości firmy Zurada.

{MAP_KNOWLEDGE}

Poniżej otrzymasz strony produktów, które przeszły już filtr bezpieczeństwa.
Opieraj się W 100% na faktach z dostarczonych stron — nie wymyślaj przeznaczenia produktu.

Oceń każdy produkt według 2 osi:

A) FORMA I TYP
   Forma (żel/piana/spray/proszek), przeznaczenie (ręczne vs maszynowe), przykłady ("np. X lub Y"
   to PRZYKŁADY — uwzględnij WSZYSTKIE produkty tego samego typu).

B) KONTEKST I SKALA
   Produkty profesjonalne rekomenduj gdy pasują. Wyklucz tylko czysto przemysłowe (beczki 200L).

Jeśli produkt ogólnie pasuje i nie ma przeciwwskazań — uwzględnij go.
Jeśli żaden nie pasuje — zwróć pustą listę product_ids."""


def recommend(question: str, product_context: str) -> tuple[RecommendModel, object]:
    result, completion = llm.chat.completions.create_with_completion(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": RECOMMEND_SYSTEM_3.format(MAP_KNOWLEDGE=MAP_KNOWLEDGE)},
            {"role": "user",   "content": f"Strony produktów (po filtrze bezpieczeństwa):\n\n{product_context}\n\n---\n\nPytanie klienta: {question}"},
        ],
        response_model=RecommendModel,
        max_tokens=4000,
        temperature=0,
    )
    return result, completion.usage

---
## Pełny pipeline

In [12]:
def qdrant_rag_pipeline(question: str) -> tuple[list[int], dict]:
    """intent → hybrid Qdrant top-20 → safety filter → reranker top-5 → recommend"""
    t0 = time.perf_counter()

    intent, usage_intent = extract_intent(question)

    product_context, product_names, rrf_scores = qdrant_retrieve(question, intent)

    safe_context, safe_names, safety_result, usage_safety = filter_unsafe(
        question, product_context, product_names
    )

    # Reranker: scores each (question, product) pair with cross-encoder
    reranked_names, reranker_scores = rerank(question, safe_names)
    reranked_context = load_product_pages(reranked_names)

    rec, usage_rec = recommend(question, reranked_context)

    elapsed      = time.perf_counter() - t0
    returned_ids = [c.product_id for c in rec.product_ids]

    stats = {
        "intent":            {"surfaces": [s.value for s in intent.surfaces],
                              "categories": [c.value for c in intent.categories],
                              "methods": [m.value for m in intent.methods]},
        "n_retrieved":       len(product_names),
        "n_safe":            len(safe_names),
        "n_reranked":        len(reranked_names),
        "excluded_ids":      safety_result.excluded_ids,
        "top_rrf_score":     rrf_scores[0] if rrf_scores else None,
        "top_reranker_score": reranker_scores[0] if reranker_scores else None,
        "prompt_tokens":     usage_intent.prompt_tokens  + usage_safety.prompt_tokens  + usage_rec.prompt_tokens,
        "completion_tokens": usage_intent.completion_tokens + usage_safety.completion_tokens + usage_rec.completion_tokens,
        "total_tokens":      usage_intent.total_tokens   + usage_safety.total_tokens   + usage_rec.total_tokens,
        "time_seconds":      round(elapsed, 2),
    }
    return returned_ids, stats


# Single-run over all questions
results = []

for q in questions:
    qid      = q["id"]
    expected = set(q["expectedProductIds"])
    try:
        returned_ids, stats = qdrant_rag_pipeline(q["question"])
        returned = set(returned_ids)
        error    = None
    except Exception as e:
        returned = set()
        stats    = {"n_retrieved": 0, "n_safe": 0, "n_reranked": 0, "excluded_ids": [],
                    "top_rrf_score": None, "top_reranker_score": None,
                    "time_seconds": 0, "prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
        error    = str(e)

    correct = returned == expected
    results.append({
        "id": qid, "difficulty": q.get("difficulty", ""),
        "expected": sorted(expected), "returned": sorted(returned),
        "correct": correct,
        "missed":  sorted(expected - returned),
        "extra":   sorted(returned - expected),
        **{k: stats[k] for k in ["n_retrieved", "n_safe", "n_reranked",
                                   "excluded_ids", "top_reranker_score",
                                   "total_tokens", "time_seconds"]},
        "intent": stats.get("intent", {}),
        "error":  error,
    })

    status = "✓" if correct else "✗"
    print(f"[{status}] Q{qid:02d} [{q['difficulty']:4s}]  "
          f"retrieved={stats['n_retrieved']:2d}→safe={stats['n_safe']:2d}→reranked={stats['n_reranked']:2d}  "
          f"top_reranker={stats['top_reranker_score'] or 0:.2f}  "
          f"tokens={stats['total_tokens']:6,}  time={stats['time_seconds']:.1f}s  "
          f"expected={sorted(expected)}  returned={sorted(returned)}")

n_correct = sum(r["correct"] for r in results)
print(f"\n{n_correct}/{len(results)} exact matches  "
      f"avg time={sum(r['time_seconds'] for r in results)/len(results):.1f}s  "
      f"avg tokens={sum(r['total_tokens'] for r in results)/len(results):,.0f}")

[✗] Q01 [easy]  retrieved=20→safe=16→reranked= 5  top_reranker=0.91  tokens=27,477  time=6.9s  expected=[12, 82, 139]  returned=[82, 139]
[✓] Q08 [easy]  retrieved=20→safe= 5→reranked= 5  top_reranker=0.46  tokens=23,219  time=5.3s  expected=[136, 137, 189, 190]  returned=[136, 137, 189, 190]
[✓] Q09 [easy]  retrieved= 8→safe= 8→reranked= 5  top_reranker=0.95  tokens=17,213  time=5.6s  expected=[233, 257]  returned=[233, 257]
[✓] Q10 [easy]  retrieved=18→safe=18→reranked= 5  top_reranker=0.99  tokens=22,460  time=5.5s  expected=[213, 217, 218]  returned=[213, 217, 218]
[✗] Q02 [easy]  retrieved= 9→safe= 8→reranked= 5  top_reranker=1.00  tokens=20,764  time=5.7s  expected=[5, 34, 128]  returned=[5, 34, 67, 128]
[✗] Q03 [hard]  retrieved=20→safe=14→reranked= 5  top_reranker=0.93  tokens=27,570  time=7.7s  expected=[45, 112, 152]  returned=[44, 122]
[✓] Q04 [hard]  retrieved=20→safe= 3→reranked= 3  top_reranker=0.47  tokens=29,608  time=7.8s  expected=[11]  returned=[11]
[✗] Q05 [easy]  r

---
## Zapis wyników (8 równoległych przebiegów)

In [13]:
N_RUNS      = 8
MAX_WORKERS = 9
OUTPUT_FILE = BASE_DIR / "reranker_rag_runs.json"


def _run_once(q: dict, run_idx: int) -> tuple[int, int, dict]:
    try:
        returned_ids, stats = qdrant_rag_pipeline(q["question"])
        result = {
            "run":                run_idx,
            "returned_ids":       returned_ids,
            "model":              LLM_MODEL,
            "n_retrieved":        stats["n_retrieved"],
            "n_safe":             stats["n_safe"],
            "n_reranked":         stats["n_reranked"],
            "excluded_ids":       stats["excluded_ids"],
            "top_reranker_score": stats["top_reranker_score"],
            "prompt_tokens":      stats["prompt_tokens"],
            "completion_tokens":  stats["completion_tokens"],
            "total_tokens":       stats["total_tokens"],
            "time_seconds":       stats["time_seconds"],
            "intent":             stats["intent"],
        }
    except Exception as e:
        result = {"run": run_idx, "error": str(e)}
    return q["id"], run_idx, result


tasks = [(q, run_idx) for q in questions for run_idx in range(1, N_RUNS + 1)]
raw: dict[int, list] = defaultdict(list)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(_run_once, q, run_idx): (q["id"], run_idx)
               for q, run_idx in tasks}
    for future in as_completed(futures):
        qid, run_idx, result = future.result()
        raw[qid].append(result)
        status = "✓" if not result.get("error") else "✗"
        print(f"  {status} Q{qid:02d} run {run_idx}")


repeated = []
for q in questions:
    qid  = q["id"]
    runs = sorted(raw[qid], key=lambda r: r["run"])

    times  = [r["time_seconds"] for r in runs if "time_seconds" in r]
    tokens = [r["total_tokens"]  for r in runs if "total_tokens"  in r]

    avg_time   = round(sum(times)  / len(times),  2) if times  else None
    avg_tokens = round(sum(tokens) / len(tokens), 0) if tokens else None

    repeated.append({
        "id":                   qid,
        "question":             q["question"],
        "expectedProductIds":   q["expectedProductIds"],
        "expectedNoProductIds": q["expectedNoProductIds"],
        "difficulty":           q.get("difficulty", ""),
        "avg_time_seconds":     avg_time,
        "avg_total_tokens":     avg_tokens,
        "runs":                 runs,
    })

    expected_set = set(q["expectedProductIds"])
    n_correct    = sum(1 for r in runs if set(r.get("returned_ids", [])) == expected_set)
    print(f"Q{qid:02d} [{q['difficulty']:4s}] {n_correct}/{N_RUNS} correct  "
          f"avg={avg_time:.1f}s  avg_tokens={avg_tokens:,.0f}")


with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump({
        "timestamp":      datetime.now().isoformat(),
        "llm_model":      LLM_MODEL,
        "embed_model":    EMBED_MODEL,
        "sparse_model":   SPARSE_MODEL,
        "reranker_model": RERANKER_MODEL,
        "reranker_top_k": RERANKER_TOP_K,
        "approach":       "reranker_rag",
        "n_runs":         N_RUNS,
        "questions":      repeated,
    }, f, ensure_ascii=False, indent=2)

print(f"\nSaved: {OUTPUT_FILE}")

  ✓ Q01 run 2
  ✓ Q01 run 7
  ✓ Q01 run 1
  ✓ Q01 run 5
  ✓ Q01 run 4
  ✓ Q01 run 3
  ✓ Q08 run 2
  ✓ Q01 run 6
  ✓ Q01 run 8
  ✓ Q08 run 3
  ✓ Q08 run 6
  ✓ Q08 run 5
  ✓ Q08 run 7
  ✓ Q08 run 1
  ✓ Q08 run 4
  ✓ Q09 run 1
  ✓ Q08 run 8
  ✓ Q09 run 2
  ✓ Q09 run 4
  ✓ Q09 run 6
  ✓ Q09 run 5
  ✓ Q09 run 3
  ✓ Q09 run 8
  ✓ Q09 run 7
  ✓ Q10 run 1
  ✓ Q10 run 2
  ✓ Q10 run 3
  ✓ Q10 run 5
  ✓ Q10 run 4
  ✓ Q10 run 7
  ✓ Q10 run 8
  ✓ Q10 run 6
  ✓ Q02 run 1
  ✓ Q02 run 2
  ✓ Q02 run 3
  ✓ Q02 run 4
  ✓ Q02 run 6
  ✓ Q02 run 5
  ✓ Q02 run 8
  ✓ Q03 run 1
  ✓ Q02 run 7
  ✓ Q03 run 3
  ✓ Q03 run 2
  ✓ Q03 run 5
  ✓ Q03 run 6
  ✓ Q03 run 7
  ✓ Q03 run 4
  ✓ Q03 run 8
  ✓ Q04 run 2
  ✓ Q04 run 1
  ✓ Q04 run 3
  ✓ Q04 run 4
  ✓ Q04 run 5
  ✓ Q04 run 7
  ✓ Q04 run 8
  ✓ Q05 run 1
  ✓ Q05 run 2
  ✓ Q04 run 6
  ✓ Q05 run 4
  ✓ Q05 run 3
  ✓ Q05 run 6
  ✓ Q05 run 5
  ✓ Q06 run 1
  ✓ Q05 run 8
  ✓ Q06 run 2
  ✓ Q05 run 7
  ✓ Q06 run 5
  ✓ Q06 run 3
  ✓ Q06 run 4
  ✓ Q06 run 6
  ✓ Q06 run 8
  ✓ Q0

In [14]:
qdrant.close()